# Ground-Truth Driver / Non-Driver Gene Retrieval (Unmatched Negatives)

Retrieves the ground-truth driver and non-driver gene sets (protein-coding universe -> driver
exclusion -> mutation-frequency filter -> disease-pathway filter) and saves **four separate
files**:

1. `driver_genes.txt` -- one gene per line, the ground-truth positive (driver) set
2. `non_driver_genes.txt` -- one gene per line, the ground-truth negative (non-driver) set
3. `unlabeled_genes.txt` -- one gene per line, protein-coding genes that ended up with
   neither label (genes outside the NCG/CGC/IntOGen/Bailey driver union, plus
   genes dropped by the mutation-frequency/pathway filters)
4. `gene_labels.csv` -- drivers + non-drivers combined into one labeled table (`label`
   column: 1 = driver, 0 = non-driver; unlabeled genes are not included in this file)

Non-driver genes are **not** length/expression-matched here -- whatever survives the
mutation-frequency and disease-pathway filters is kept as-is as the negative class.


In [1]:
import os
import random
import sys
from pathlib import Path

import pandas as pd

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import config
from src.driver_labeling import (
    apply_mutation_frequency_filter,
    apply_pathway_filter,
    build_gene_labels,
)
from src.gene_universe import build_protein_coding_gene_universe
from src.negative_sampling import build_driver_exclusion_set
from src.pathway_filter import build_disease_pathway_genes

random.seed(config.RANDOM_SEED)
config.DATA_DIR.mkdir(parents=True, exist_ok=True)
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"[CONFIG] USE_MUTATION_FILTER: {config.USE_MUTATION_FILTER}")
print(f"[CONFIG] USE_PATHWAY_FILTER: {config.USE_PATHWAY_FILTER}")


[CONFIG] USE_MUTATION_FILTER: True
[CONFIG] USE_PATHWAY_FILTER: True


## 1. Protein-coding gene universe

Parses the GENCODE GTF and keeps only `gene_type == "protein_coding"`.


In [2]:
all_genes = build_protein_coding_gene_universe(
    "../../data/gencode.v49.basic.annotation.gtf"
)
# Use a single symbol convention for GENCODE, CGC, TCGA, and Reactome joins.
all_genes = {gene.upper() for gene in all_genes}
pd.DataFrame(sorted(all_genes), columns=["gene_name"]).to_csv(
    config.GENCODE_GENES_FILE, index=False
)
print(f"[GENE UNIVERSE] {len(all_genes)} protein-coding genes")


[GENE UNIVERSE] 20070 protein-coding genes


## 2. Ground-truth driver genes and the expanded exclusion set

Positives are the union of every available driver reference: NCG, CGC, IntOGen, and Bailey
et al. 2018. The same union is excluded when deriving raw non-driver candidates, so a gene
with driver evidence in any supplied source receives a positive label rather than being
treated as an unlabeled or negative gene.


In [3]:
driver_exclusion_set = build_driver_exclusion_set(
    ncg_file=config.NCG_FILE if config.NCG_FILE.exists() else None,
    cgc_file=config.CGC_CENSUS_FILE if config.CGC_CENSUS_FILE.exists() else None,
    intogen_file=config.INTOGEN_FILE if config.INTOGEN_FILE.exists() else None,
    bailey_file=config.BAILEY_FILE if config.BAILEY_FILE.exists() else None,
)

# Label as drivers every reference gene that is present in the protein-coding universe.
driver_genes = all_genes & {gene.upper() for gene in driver_exclusion_set}
print(
    f"[DRIVERS] {len(driver_genes)} protein-coding genes labeled positive "
    "(NCG/CGC/IntOGen/Bailey union)"
)

non_drivers = all_genes - driver_genes
print(
    f"[NON-DRIVERS] {len(non_drivers)} raw candidates "
    "(excluded from NCG/CGC/IntOGen/Bailey)"
)


## 3. Mutation-frequency filter

Excludes any candidate with mutation frequency >= `config.MUTATION_FREQUENCY_THRESHOLD` in any
TCGA cancer type. Skipped entirely if `config.USE_MUTATION_FILTER` is `False`.


In [4]:
TARGET_NEGATIVE_COUNT = 2421
# 3.1% admits the 3.0%-frequency boundary group, leaving enough candidates
# to draw the target 2,421 negatives while retaining all other exclusions.
MAX_MUTATION_FREQUENCY = 0.031
pathway_mapping_file = Path("../data/processed/reactome_human_gene_pathway_mapping.tsv")

pathway_mapping = pd.read_csv(pathway_mapping_file, sep="\t", dtype=str)
mapped_pathway_genes = set(
    pathway_mapping.iloc[:, 1:].stack().dropna().str.upper()
)
print(f"[PATHWAY MAPPING] {len(mapped_pathway_genes)} genes map to graph pathways")

before = len(non_drivers)
non_drivers = {
    gene.upper() for gene in non_drivers
    if gene.upper() in mapped_pathway_genes
}
print(
    f"[PATHWAY MAPPING] {len(non_drivers)} eligible candidates "
    f"(-{before - len(non_drivers)} unmapped)"
)

before = len(non_drivers)
non_drivers = apply_mutation_frequency_filter(
    non_drivers,
    config.MUTATION_FREQUENCY_FILE,
    threshold=MAX_MUTATION_FREQUENCY,
)
print(
    f"[MUTATION FILTER] {len(non_drivers)} remaining at "
    f"< {MAX_MUTATION_FREQUENCY:.1%} (-{before - len(non_drivers)})"
)


## 4. Disease-pathway filter

Excludes candidates belonging to a Reactome pathway that is a descendant of the top-level
*Disease* pathway. Skipped entirely if `config.USE_PATHWAY_FILTER` is `False`.


In [5]:
if config.USE_PATHWAY_FILTER:
    pathway_genes = build_disease_pathway_genes(
        "../../data/reactome/ReactomePathways.gmt",
        "../../data/reactome/reactome_relations.csv",
    )
    pd.DataFrame(sorted(pathway_genes), columns=["gene"]).to_csv(
        config.DISEASE_PATHWAY_GENES_FILE, index=False
    )
    print(f"[PATHWAY FILTER] {len(pathway_genes)} disease-pathway genes")

    before = len(non_drivers)
    non_drivers = apply_pathway_filter(non_drivers, pathway_genes)
    print(
        f"[PATHWAY FILTER] {len(non_drivers)} remaining "
        f"(-{before - len(non_drivers)})"
    )
else:
    print("[PATHWAY FILTER] skipped (config.USE_PATHWAY_FILTER is False)")


[PATHWAY FILTER] kept 782 disease-related pathways, skipped 2048 unrelated pathways
[PATHWAY FILTER] 2505 disease-pathway genes
[PATHWAY FILTER] 2484 remaining (-553)


## 5. Save four separate files

`driver_genes.txt`, `non_driver_genes.txt`, and `unlabeled_genes.txt` each hold one gene set
on its own (one gene per line); `gene_labels.csv` combines drivers + non-drivers into a
single labeled table for anything downstream that wants the pair together (unlabeled genes
are intentionally left out of that file since they have no `label` value).


In [6]:
selected_negative_count = min(TARGET_NEGATIVE_COUNT, len(non_drivers))
if selected_negative_count < TARGET_NEGATIVE_COUNT:
    print(
        f"[NEGATIVE SELECTION] Only {selected_negative_count} eligible "
        f"pathway-mapped negatives are available; using all of them instead of "
        f"the requested {TARGET_NEGATIVE_COUNT}."
    )

non_drivers = set(
    random.Random(config.RANDOM_SEED).sample(
        sorted(non_drivers), selected_negative_count
    )
)
print(f"[NEGATIVE SELECTION] Selected {len(non_drivers)} pathway-mapped negatives")

# Every protein-coding gene outside the supervised positive/negative sets is unlabeled.
unlabeled_genes = all_genes - driver_genes - non_drivers
print(f"[UNLABELED] {len(unlabeled_genes)} genes excluded from both classes")

driver_path = config.PROCESSED_DIR / "driver_genes.txt"
nondriver_path = config.PROCESSED_DIR / "non_driver_genes.txt"
unlabeled_path = config.PROCESSED_DIR / "unlabeled_genes.txt"

driver_path.write_text("\n".join(sorted(driver_genes)) + "\n")
nondriver_path.write_text("\n".join(sorted(non_drivers)) + "\n")
unlabeled_path.write_text("\n".join(sorted(unlabeled_genes)) + "\n")

labels_df = build_gene_labels(driver_genes, non_drivers)
print(labels_df["label"].value_counts())
labels_df.to_csv(config.GENE_LABELS_FILE, index=False)

print(f"[DONE] Drivers ({len(driver_genes)}): {driver_path}")
print(f"[DONE] Non-drivers ({len(non_drivers)}, pathway-mapped): {nondriver_path}")
print(f"[DONE] Unlabeled ({len(unlabeled_genes)}): {unlabeled_path}")
print(f"[DONE] Combined labels (drivers + non-drivers only): {config.GENE_LABELS_FILE}")

labels_df.head()


[NEGATIVE SELECTION] Selected 2421 pathway-mapped negatives
[UNLABELED] 16905 genes excluded from both classes
label
0    2421
1     763
Name: count, dtype: int64
[DONE] Drivers (763): ../data/processed/driver_genes.txt
[DONE] Non-drivers (2421, pathway-mapped): ../data/processed/non_driver_genes.txt
[DONE] Unlabeled (16905): ../data/processed/unlabeled_genes.txt
[DONE] Combined labels (drivers + non-drivers only): ../data/processed/gene_labels_driver_vs_nondriver.csv


,gene,label
0,BCL7A,1
1,TCF7L2,1
2,PDGFRB,1
3,THRAP3,1
4,NT5C2,1
